In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

items = spark.table(f"{catalog_name}.{gold_schema}.fact_order_items")
products = spark.table(f"{catalog_name}.{gold_schema}.dim_product")

# bashkon artikujt me kategorin e produktit, grupon sipas dites dhe kategorise, dhe llogarit totalet (te ardhura, transport, numer artikujsh, numer porosish
agg = (items
    .join(products.select("product_id", "product_category"), on="product_id", how="left")
    .groupBy("date_key", "product_category")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.sum("freight_value").alias("total_freight"),
        F.count("*").alias("item_count"),
        F.countDistinct("order_id").alias("order_count")
    ))

(agg.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.agg_daily_category"))
print(f"Wrote {agg.count():,} aggregate rows")